Unless explicitly stated otherwise all files in this repository are licensed under the Apache-2.0 License.

This product includes software developed at Datadog (https://www.datadoghq.com/)
Copyright 2026 Datadog, Inc.

# Toto 2.0 Quick Start

Forecast time series with Toto 2.0 in a few lines of code.

## Installation

```bash
pip install "toto-2 @ git+https://github.com/DataDog/toto.git#subdirectory=toto2"
```

This also installs `dd-unit-scaling` and all other dependencies automatically.

## Load a pretrained model

In [1]:
import torch
from toto2 import Toto2Model

SIZE = "4m"  # 4m | 22m | 313m | 1B | 2.5B
CHECKPOINT = f"Datadog/Toto-2.0-{SIZE}"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Toto2Model.from_pretrained(CHECKPOINT, map_location=device)
model = model.to(device).eval()

print(f"Loaded {CHECKPOINT}: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Patch size: {model.config.patch_size}")

ImportError: cannot import name 'Toto2Model' from 'toto2' (unknown location)

## Forecast with raw tensors

`model.forecast()` takes a dict with `target`, `target_mask`, and `series_ids`
and returns a quantile tensor of shape `(Q, batch, n_var, horizon)` where Q=9
corresponds to quantile levels [0.1, 0.2, ..., 0.9].

In [ ]:
# Synthetic univariate series: trend + seasonality + noise
context_length = 512
t = torch.arange(context_length, dtype=torch.float32)
series = 100 + 0.05 * t + 10 * torch.sin(2 * torch.pi * t / 24) + torch.randn(context_length)

# Shape: (batch=1, n_var=1, time)
target = series.unsqueeze(0).unsqueeze(0).to(device)
target_mask = torch.ones_like(target, dtype=torch.bool)
series_ids = torch.zeros(1, 1, dtype=torch.long, device=device)

horizon = 96
quantiles = model.forecast(
    {"target": target, "target_mask": target_mask, "series_ids": series_ids},
    horizon=horizon,
)

print(f"Output shape: {quantiles.shape}")  # (9, 1, 1, 96)
print(f"Quantile levels: {model.output_head.knots}")

In [ ]:
import matplotlib.pyplot as plt

median = quantiles[4, 0, 0].cpu()  # 0.5 quantile
q10 = quantiles[0, 0, 0].cpu()     # 0.1 quantile
q90 = quantiles[8, 0, 0].cpu()     # 0.9 quantile

fig, ax = plt.subplots(figsize=(12, 4))
ctx = series[-96:].cpu()
ax.plot(range(96), ctx, label="Context", color="black")
ax.plot(range(96, 96 + horizon), median, label="Median forecast", color="tab:blue")
ax.fill_between(range(96, 96 + horizon), q10, q90, alpha=0.2, color="tab:blue", label="80% interval")
ax.legend()
ax.set_title("Toto 2.0 Forecast")
plt.tight_layout()
plt.show()

## Multivariate forecasting

Pass multiple variates along the `n_var` dimension. The model applies
alternating time and variate attention layers.

In [ ]:
n_var = 3
target_mv = torch.randn(1, n_var, 512, device=device)
mask_mv = torch.ones(1, n_var, 512, dtype=torch.bool, device=device)
ids_mv = torch.zeros(1, n_var, dtype=torch.long, device=device)

quantiles_mv = model.forecast(
    {"target": target_mv, "target_mask": mask_mv, "series_ids": ids_mv},
    horizon=48,
)
print(f"Multivariate output: {quantiles_mv.shape}")  # (9, 1, 3, 48)

## Handling missing values

Set `target_mask` to `False` where observations are missing.
The model masks these in both the scaler and attention.

In [ ]:
target_missing = series.unsqueeze(0).unsqueeze(0).to(device)
mask_missing = torch.ones_like(target_missing, dtype=torch.bool)
# Drop 20% of observations at random
mask_missing[0, 0, torch.randperm(context_length)[:context_length // 5]] = False

quantiles_missing = model.forecast(
    {
        "target": target_missing,
        "target_mask": mask_missing,
        "series_ids": torch.zeros(1, 1, dtype=torch.long, device=device),
    },
    horizon=96,
)
print(f"With missing values: {quantiles_missing.shape}")